# Customer Churn Prediction

**Final year project | Machine learning classification**

This project studies whether a telecom customer's account, services, and billing profile can help predict churn. It compares Logistic Regression, Decision Tree, and Random Forest models and evaluates the selected pipeline on a held-out test set.

> Run this notebook from the project folder after installing `requirements.txt`. The training workflow also writes the reusable model and project reports.


## Abstract

Customer churn describes customers ending a service relationship. This project builds a supervised binary classifier using the supplied Telco customer dataset. The notebook explores the data, handles blank total charges, compares class-weighted models with stratified cross-validation, and reports holdout metrics that account for the smaller churn class. A saved pipeline and a Streamlit form demonstrate how a new profile can be scored.

The evaluation describes performance on this dataset split; it does not prove that acting on a prediction will reduce churn.


## Objectives

1. Prepare customer records for a supervised learning task.
2. Compare three interpretable, commonly used classifiers using ROC-AUC and supporting metrics.
3. Evaluate the selected model on a separate stratified holdout set.
4. Package preprocessing and classification into one reusable pipeline.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "train_model.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Customer_Churn_Final_Year_Project folder.")
sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "telco_customer_churn.csv"
print("Project:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

## 1. Dataset overview

The source file contains one row per customer. `Churn` is the target, and `customerID` is excluded from prediction because it is only an identifier. The original dataset has 7,043 rows and 19 candidate inputs.


In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
print("Target counts:")
display(df["Churn"].value_counts(dropna=False).rename_axis("Churn").to_frame("Customers"))
print("Blank TotalCharges rows:", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Duplicate rows:", df.duplicated().sum())

## 2. Exploratory analysis

Churn is less frequent than non-churn. Accuracy alone can therefore hide poor detection of customers who leave. The plots below summarize the class balance and two predictors that are often useful to inspect.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["Churn"].value_counts().reindex(["No", "Yes"]).plot(kind="bar", ax=axes[0], color=["#94a3b8", "#2563eb"])
axes[0].set_title("Customer churn counts")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Customers")

contract_churn = pd.crosstab(df["Contract"], df["Churn"], normalize="index").reindex(columns=["No", "Yes"])
contract_churn.plot(kind="bar", stacked=True, ax=axes[1], color=["#94a3b8", "#2563eb"])
axes[1].set_title("Churn share by contract")
axes[1].set_ylabel("Share within contract")
axes[1].legend(title="Churn")
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
display(df[numeric_columns].describe().T)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, column in zip(axes, numeric_columns):
    pd.to_numeric(df[column], errors="coerce").plot(kind="hist", bins=25, ax=ax, color="#0f766e")
    ax.set_title(column)
    ax.set_xlabel(column)
plt.tight_layout()
plt.show()

## 3. Preprocessing and validation design

The project uses a stratified 80/20 train/test split. Imputation, scaling, and one-hot encoding live inside each model pipeline, so each validation fold learns those transformations only from its own training portion. Class weights address the imbalanced target without creating synthetic examples before cross-validation.

Candidate pipelines are compared with five-fold stratified cross-validation on the training split. The model with the highest mean ROC-AUC is refitted on all training rows. The test split is used once for the final report.


In [ ]:
from src.train_model import train

results = train(DATA_PATH, PROJECT_ROOT, cv_folds=5)
print("Selected model:", results["selected_model"])

## 4. Cross-validation comparison

Each row below reports the mean and standard deviation across training folds. ROC-AUC is used to select the candidate; recall, precision, and F1 make the churn-class trade-offs visible.


In [ ]:
cv_results = pd.DataFrame(results["cross_validation"])
display(cv_results[["model", "roc_auc_mean", "roc_auc_std", "recall_mean", "precision_mean", "f1_mean", "accuracy_mean"]].round(3))
from IPython.display import Image, display as ipy_display
ipy_display(Image(filename=str(PROJECT_ROOT / "reports" / "cv_model_comparison.png")))

## 5. Holdout evaluation

The final metrics below describe only the untouched test split. Precision answers how many predicted churners actually churned in this sample; recall answers how many actual churners were found. ROC-AUC summarizes ranking across thresholds, while PR-AUC focuses on the minority positive class.


In [ ]:
test_metrics = results["holdout_test"]
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]:
    print(f"{metric:>10}: {test_metrics[metric]:.3f}")
ipy_display(Image(filename=str(PROJECT_ROOT / "reports" / "holdout_confusion_matrix.png")))

In [ ]:
importance_path = PROJECT_ROOT / "reports" / "feature_importance.csv"
if importance_path.exists():
    print("Top global feature signals (not causal effects):")
    display(pd.read_csv(importance_path).head(10))
    ipy_display(Image(filename=str(PROJECT_ROOT / "reports" / "feature_importance.png")))
else:
    print("This selected estimator does not expose native feature importance.")

## 6. Score one customer profile

The saved artifact includes both preprocessing and the classifier. Raw category labels are accepted, and unseen categories are safely ignored by the encoder. The displayed score is an uncalibrated model score; use it for the demonstration rather than interpreting it as a precise individual probability.


In [ ]:
from src.predict import load_artifact, predict_one

artifact = load_artifact(PROJECT_ROOT / "artifacts" / "churn_pipeline.joblib")
example_customer = {
    "gender": "Female", "SeniorCitizen": 0, "Partner": "Yes", "Dependents": "No",
    "tenure": 1, "PhoneService": "No", "MultipleLines": "No phone service",
    "InternetService": "DSL", "OnlineSecurity": "No", "OnlineBackup": "Yes",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 29.85, "TotalCharges": 29.85
}
label, score = predict_one(example_customer, artifact)
print(f"Prediction: {label}")
print(f"Estimated churn score: {score:.1%}")

## Conclusion and limitations

This project demonstrates a reproducible churn classification workflow and creates a saved model plus an interactive app. For the project report, copy the actual holdout and cross-validation results from the generated JSON and figures. Avoid claiming that the model causes retention: the dataset is historical and observational, a single split is not external validation, the scores are not calibrated, and the costs of false alarms versus missed churn are not specified.

Possible next steps include testing on newer data, calibration, threshold selection from explicit business costs, drift monitoring, and privacy/fairness review before deployment.
